# MLP, GNN и GCN для Facebook Page-Page

В ноутбуке реализована лабораторная работа по классификации узлов графа Facebook Page-Page. Сравниваются три подхода:

- `MLP`, который использует только признаки узлов;
- `GraphSAGE` как пример графовой нейронной сети общего вида с агрегацией сообщений от соседей;
- `GCN`, который использует графовую сверточную агрегацию.

Для всех моделей используется одно и то же стратифицированное разбиение узлов на train/validation/test и единый цикл обучения.

## Подготовка окружения

Если зависимости ещё не установлены, выполните следующую ячейку, затем перезапустите kernel и запустите ноутбук сверху вниз.

In [ ]:
# Раскомментируйте строку при первом запуске ноутбука.
# %pip install -r requirements.txt

In [ ]:
import copy
import importlib.util
import random
import warnings
from pathlib import Path

required_packages = {
    "torch": "torch",
    "torch_geometric": "torch_geometric",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "seaborn": "seaborn",
}

missing_packages = [package_name for import_name, package_name in required_packages.items() if importlib.util.find_spec(import_name) is None]
if missing_packages:
    raise ImportError(
        "Не установлены зависимости: "
        + ", ".join(missing_packages)
        + ". Выполните `%pip install -r requirements.txt` и перезапустите kernel."
    )

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch_geometric.datasets import FacebookPagePage
from torch_geometric.nn import GCNConv, SAGEConv

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid")

In [ ]:
SEED = 42
DATA_DIR = Path("data")
EPOCHS = 200
PATIENCE = 30
HIDDEN_CHANNELS = 64
DROPOUT = 0.5
LEARNING_RATE = 0.01
WEIGHT_DECAY = 5e-4


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Загрузка и первичный анализ данных

`FacebookPagePage` загружается через `torch_geometric.datasets`. При первом запуске PyG скачает данные в локальную папку `data/`.

In [ ]:
def prepare_facebook_npz_from_musae(raw_dir: Path) -> None:
    """Build the PyG-compatible NPZ file if the default PyG host is unavailable."""
    import json
    import shutil
    from urllib.request import Request, urlopen

    raw_dir.mkdir(parents=True, exist_ok=True)
    sources = {
        "musae_facebook_edges.csv": "https://github.com/user-attachments/files/23413544/musae_facebook_edges.csv",
        "musae_facebook_features.json": "https://github.com/user-attachments/files/23413543/musae_facebook_features.json",
        "musae_facebook_target.csv": "https://github.com/user-attachments/files/23413542/musae_facebook_target.csv",
    }

    for filename, url in sources.items():
        target_path = raw_dir / filename
        if target_path.exists():
            continue
        request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urlopen(request, timeout=120) as response:
            target_path.write_bytes(response.read())

    edges_df = pd.read_csv(raw_dir / "musae_facebook_edges.csv")
    target_df = pd.read_csv(raw_dir / "musae_facebook_target.csv")
    features_by_node = json.loads((raw_dir / "musae_facebook_features.json").read_text(encoding="utf-8"))

    node_ids = target_df["id"].astype(int).to_numpy()
    node_id_to_position = {node_id: position for position, node_id in enumerate(node_ids)}

    max_feature_id = max(max(feature_ids) for feature_ids in features_by_node.values() if feature_ids)
    features = np.zeros((len(node_ids), max_feature_id + 1), dtype=np.float32)
    for node_id_str, feature_ids in features_by_node.items():
        features[node_id_to_position[int(node_id_str)], feature_ids] = 1.0

    target = target_df["page_type"].astype("category").cat.codes.to_numpy(dtype=np.int64)
    source_col, target_col = edges_df.columns[:2]
    directed_edges = edges_df[[source_col, target_col]].astype(int).replace(node_id_to_position).to_numpy(dtype=np.int64)
    undirected_edges = np.vstack([directed_edges, directed_edges[:, ::-1]])

    np.savez(raw_dir / "facebook.npz", features=features, target=target, edges=undirected_edges)

    processed_dir = raw_dir.parent / "processed"
    if processed_dir.exists():
        shutil.rmtree(processed_dir)


def load_facebook_page_page(root: Path) -> FacebookPagePage:
    try:
        dataset = FacebookPagePage(root=str(root))
        if dataset[0].num_edges < 300_000 and (root / "raw" / "musae_facebook_edges.csv").exists():
            prepare_facebook_npz_from_musae(root / "raw")
            dataset = FacebookPagePage(root=str(root), force_reload=True)
        return dataset
    except Exception as error:
        print("Стандартная загрузка PyG не удалась, используется fallback из MUSAE-файлов:", error)
        prepare_facebook_npz_from_musae(root / "raw")
        return FacebookPagePage(root=str(root), force_reload=True)


dataset = load_facebook_page_page(DATA_DIR / "FacebookPagePage")
data = dataset[0]

num_classes = int(data.y.max().item() + 1)
class_counts = pd.Series(data.y.cpu().numpy()).value_counts().sort_index()
class_distribution = pd.DataFrame({"class": class_counts.index, "nodes": class_counts.values})
class_distribution["share"] = class_distribution["nodes"] / class_distribution["nodes"].sum()

summary = pd.DataFrame(
    {
        "metric": [
            "Количество узлов",
            "Количество ребер",
            "Размерность признаков",
            "Количество классов",
            "Средняя степень",
        ],
        "value": [
            data.num_nodes,
            data.num_edges,
            dataset.num_node_features,
            num_classes,
            round(data.num_edges / data.num_nodes, 2),
        ],
    }
)

display(summary)
display(class_distribution)

plt.figure(figsize=(8, 4))
sns.barplot(data=class_distribution, x="class", y="nodes", color="#4C72B0")
plt.title("Распределение классов Facebook Page-Page")
plt.xlabel("Класс")
plt.ylabel("Количество узлов")
plt.show()

## Разбиение на train, validation и test

Разбиение делается стратифицированно по меткам узлов. Это важно, потому что все три модели должны сравниваться на одних и тех же объектах.

In [ ]:
def indices_to_mask(indices: np.ndarray, size: int) -> torch.Tensor:
    mask = torch.zeros(size, dtype=torch.bool)
    mask[indices] = True
    return mask


y = data.y.cpu().numpy()
all_indices = np.arange(data.num_nodes)

train_idx, temp_idx = train_test_split(
    all_indices,
    test_size=0.4,
    random_state=SEED,
    stratify=y,
)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    random_state=SEED,
    stratify=y[temp_idx],
)

data.train_mask = indices_to_mask(train_idx, data.num_nodes)
data.val_mask = indices_to_mask(val_idx, data.num_nodes)
data.test_mask = indices_to_mask(test_idx, data.num_nodes)

def split_distribution(mask: torch.Tensor, split_name: str) -> pd.DataFrame:
    counts = pd.Series(data.y[mask].cpu().numpy()).value_counts().sort_index()
    frame = pd.DataFrame({"class": counts.index, "nodes": counts.values})
    frame["split"] = split_name
    frame["share"] = frame["nodes"] / frame["nodes"].sum()
    return frame

split_stats = pd.concat(
    [
        split_distribution(data.train_mask, "train"),
        split_distribution(data.val_mask, "validation"),
        split_distribution(data.test_mask, "test"),
    ],
    ignore_index=True,
)

display(
    pd.DataFrame(
        {
            "split": ["train", "validation", "test"],
            "nodes": [int(data.train_mask.sum()), int(data.val_mask.sum()), int(data.test_mask.sum())],
        }
    )
)

plt.figure(figsize=(10, 4))
sns.barplot(data=split_stats, x="class", y="share", hue="split")
plt.title("Доли классов в разбиениях")
plt.xlabel("Класс")
plt.ylabel("Доля")
plt.show()

data = data.to(device)

## Архитектуры моделей

MLP получает только матрицу признаков `x`. GraphSAGE и GCN дополнительно используют `edge_index`, поэтому учитывают структуру графа.

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_channels: int, hidden_channels: int, out_channels: int, dropout: float = 0.5):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_channels, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, out_channels),
        )

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor | None = None) -> torch.Tensor:
        return self.layers(x)


class GraphSAGEGNN(nn.Module):
    def __init__(self, in_channels: int, hidden_channels: int, out_channels: int, dropout: float = 0.5):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x


class GCN(nn.Module):
    def __init__(self, in_channels: int, hidden_channels: int, out_channels: int, dropout: float = 0.5):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x


def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

## Обучение и оценка

Все модели оптимизируются `Adam` с одинаковыми гиперпараметрами. Лучшая версия модели выбирается по `macro F1` на валидации; при отсутствии улучшения используется ранняя остановка.

In [ ]:
def get_logits(model: nn.Module, graph_data) -> torch.Tensor:
    return model(graph_data.x, graph_data.edge_index)


@torch.no_grad()
def evaluate(model: nn.Module, graph_data, mask: torch.Tensor, criterion: nn.Module) -> dict[str, float]:
    model.eval()
    logits = get_logits(model, graph_data)
    loss = criterion(logits[mask], graph_data.y[mask]).item()
    predictions = logits[mask].argmax(dim=1).cpu().numpy()
    targets = graph_data.y[mask].cpu().numpy()
    return {
        "loss": loss,
        "accuracy": accuracy_score(targets, predictions),
        "macro_f1": f1_score(targets, predictions, average="macro", zero_division=0),
        "weighted_f1": f1_score(targets, predictions, average="weighted", zero_division=0),
    }


def train_model(
    model: nn.Module,
    graph_data,
    epochs: int = EPOCHS,
    patience: int = PATIENCE,
    lr: float = LEARNING_RATE,
    weight_decay: float = WEIGHT_DECAY,
) -> tuple[nn.Module, pd.DataFrame, dict[str, float]]:
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = []
    best_state = copy.deepcopy(model.state_dict())
    best_val_macro_f1 = -1.0
    best_val_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = get_logits(model, graph_data)
        train_loss = criterion(logits[graph_data.train_mask], graph_data.y[graph_data.train_mask])
        train_loss.backward()
        optimizer.step()

        train_metrics = evaluate(model, graph_data, graph_data.train_mask, criterion)
        val_metrics = evaluate(model, graph_data, graph_data.val_mask, criterion)
        row = {"epoch": epoch}
        row.update({f"train_{key}": value for key, value in train_metrics.items()})
        row.update({f"val_{key}": value for key, value in val_metrics.items()})
        history.append(row)

        improved = (val_metrics["macro_f1"] > best_val_macro_f1) or (
            np.isclose(val_metrics["macro_f1"], best_val_macro_f1) and val_metrics["loss"] < best_val_loss
        )
        if improved:
            best_val_macro_f1 = val_metrics["macro_f1"]
            best_val_loss = val_metrics["loss"]
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_state)
    test_metrics = evaluate(model, graph_data, graph_data.test_mask, criterion)
    return model, pd.DataFrame(history), test_metrics

In [ ]:
set_seed(SEED)
models = {
    "MLP": MLP(dataset.num_node_features, HIDDEN_CHANNELS, num_classes, DROPOUT),
    "GraphSAGE": GraphSAGEGNN(dataset.num_node_features, HIDDEN_CHANNELS, num_classes, DROPOUT),
    "GCN": GCN(dataset.num_node_features, HIDDEN_CHANNELS, num_classes, DROPOUT),
}

trained_models = {}
histories = {}
results = []

for model_name, model in models.items():
    print(f"Обучение {model_name}...")
    model = model.to(device)
    trained_model, history, test_metrics = train_model(model, data)
    trained_models[model_name] = trained_model
    histories[model_name] = history.assign(model=model_name)

    best_epoch_idx = history["val_macro_f1"].idxmax()
    best_epoch = history.loc[best_epoch_idx]
    results.append(
        {
            "model": model_name,
            "parameters": count_parameters(trained_model),
            "epochs": int(history["epoch"].max()),
            "best_epoch": int(best_epoch["epoch"]),
            "best_val_macro_f1": best_epoch["val_macro_f1"],
            "best_val_accuracy": best_epoch["val_accuracy"],
            "test_accuracy": test_metrics["accuracy"],
            "test_macro_f1": test_metrics["macro_f1"],
            "test_weighted_f1": test_metrics["weighted_f1"],
            "test_loss": test_metrics["loss"],
        }
    )

comparison = pd.DataFrame(results).sort_values("test_macro_f1", ascending=False).reset_index(drop=True)
display(comparison)

## Визуализация динамики обучения

In [ ]:
history_df = pd.concat(histories.values(), ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.lineplot(data=history_df, x="epoch", y="train_loss", hue="model", ax=axes[0], linestyle="--")
sns.lineplot(data=history_df, x="epoch", y="val_loss", hue="model", ax=axes[0])
axes[0].set_title("Функция потерь")
axes[0].set_xlabel("Эпоха")
axes[0].set_ylabel("Loss")

sns.lineplot(data=history_df, x="epoch", y="train_macro_f1", hue="model", ax=axes[1], linestyle="--")
sns.lineplot(data=history_df, x="epoch", y="val_macro_f1", hue="model", ax=axes[1])
axes[1].set_title("Macro F1")
axes[1].set_xlabel("Эпоха")
axes[1].set_ylabel("Macro F1")

plt.tight_layout()
plt.show()

## Матрица ошибок и отчёт для лучшей модели

In [ ]:
best_model_name = comparison.iloc[0]["model"]
best_model = trained_models[best_model_name]

best_model.eval()
with torch.no_grad():
    logits = get_logits(best_model, data)
    test_predictions = logits[data.test_mask].argmax(dim=1).cpu().numpy()
    test_targets = data.y[data.test_mask].cpu().numpy()

print(f"Лучшая модель по test macro F1: {best_model_name}")
print(classification_report(test_targets, test_predictions, zero_division=0))

cm = confusion_matrix(test_targets, test_predictions)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title(f"Матрица ошибок: {best_model_name}")
plt.xlabel("Предсказанный класс")
plt.ylabel("Истинный класс")
plt.show()

## Выводы

После запуска всех ячеек заполните выводы на основе таблицы `comparison`, графиков обучения и матрицы ошибок.

Ориентиры для анализа:

- если GraphSAGE или GCN заметно превосходят MLP, значит структура связей между страницами даёт полезный сигнал для классификации;
- если MLP близок к графовым моделям, признаки узлов сами по себе достаточно информативны;
- разница между GraphSAGE и GCN показывает, какая схема агрегации соседей лучше подходит этому графу;
- по графикам loss и macro F1 можно оценить переобучение, скорость сходимости и устойчивость моделей.